# Práctica A: ¿Puede la IA predecir el clima? Benchmark Punchana (SENAMHI Iquitos)

Este notebook implementa el caso de estudio **Caso A**:
- Evaluación rigurosa sobre **2,557 días** reales de la estación meteorológica Punchana (Loreto, Perú).
- Split temporal ciego de **90 días de holdout** (Octubre a Diciembre 2025).
- Comparativa de 8 modelos: Naive estacional, ARIMA, Holt-Winters, Random Forest, ExtraTrees, LSTM, Chronos Zero-Shot (Amazon) y TimesFM Zero-Shot (Google).
- Prueba formal de significancia estadística de **Diebold-Mariano** y regla de parsimonia.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Cargar benchmark consolidado de 8 modelos
df_bm = pd.read_csv('../data/caso_1/benchmark_punchana.csv')
print('=== BENCHMARK DE 8 MODELOS (HOLDOUT 90 DÍAS) ===')
df_bm

## 2. Visualización Comparativa del Benchmark
Observemos cómo ExtraTrees, Random Forest, ARIMA y TimesFM empatan estadísticamente (p > 0.05), mientras que los baselines y LSTM pierden significativamente.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#10b981' if p > 0.05 else '#f59e0b' if p == 1.0 else '#ef4444' for p in df_bm['p_valor_vs_mejor']]
bars = ax.barh(df_bm['modelo'], df_bm['rmse_1d'], color=colors)
ax.set_xlabel('RMSE (°C) en Holdout 90 días')
ax.set_title('Precisión de Pronóstico a 1 Día: Modelos Tradicionales vs Foundation Models')
ax.grid(axis='x', linestyle='--', alpha=0.5)
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.03, bar.get_y() + bar.get_height()/2, f'{width:.3f} °C', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Pronóstico Multi-Modelo a 7 Días con Bandas de Incertidumbre P10/P90

In [ ]:
df_fc = pd.read_csv('../data/caso_1/senamhi_forecast_7d.csv')
print('Pronóstico a 7 días:')
df_fc

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_fc['fecha'], df_fc['tmax_observada'], 'k-o', label='Observada (SENAMHI)', linewidth=2.5)
plt.plot(df_fc['fecha'], df_fc['pred_extratrees'], 'g--s', label='ExtraTrees (Ganador CPU)', linewidth=2)
plt.plot(df_fc['fecha'], df_fc['pred_timesfm'], 'b--^', label='TimesFM Zero-Shot (Google)', linewidth=1.5)
plt.plot(df_fc['fecha'], df_fc['pred_chronos'], 'm--d', label='Chronos Zero-Shot (Amazon)', linewidth=1.5)
plt.fill_between(df_fc['fecha'], df_fc['ci_lower_p10'], df_fc['ci_upper_p90'], color='green', alpha=0.15, label='Banda ExtraTrees P10-P90')
plt.title('Pronóstico Multi-Modelo Últimos 7 Días vs Verdad Terreno')
plt.ylabel('Temperatura Máxima (°C)')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()